# 03 — Multivariate Pipeline
**Цель**: модель на основе ТОЛЬКО внешних косвенных факторов (без лагов целевой).
Логика: MultFEx COVID-19 [Терехова, 2022] + замена LR → XGBoost.

Этапы: сборка датасета → RFE → Train Data Reduction → LR vs RF vs XGBoost

## 0. Зависимости

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from warnings import filterwarnings; filterwarnings("ignore")
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from utils import (compute_metrics, metrics_table, smooth_series, mark_omicron,
                   shift_features, fill_missing, train_test_split_temporal,
                   reduce_train_size, plot_forecast, plot_metrics_bar,
                   plot_train_reduction, set_plot_style)
set_plot_style()
FIGURES = "../results/figures"
METRICS = "../results/metrics"
TEST_DAYS = 31
SHIFT = 4
print("Зависимости загружены ✓")

## 1. Формирование датасета
> **Ключевой принцип**: `new_cases_smooth` НЕ используется как признак.
> Все признаки — внешние факторы, сдвинутые на `shift=4` дня (инкубационный период).

In [ ]:
# ─── Реальная сборка ───
# df = df_covid.merge(df_weather, on="date")
#              .merge(df_mobility, on="date")
#              .merge(df_yandex, on="date")
#              .merge(df_stringency, on="date")

# ─── Заглушка ───
date_range = pd.date_range("2021-01-01", "2023-06-30", freq="D")
np.random.seed(42)
n = len(date_range)
trend = np.concatenate([np.exp(np.linspace(4,6,365)),
                        np.exp(np.linspace(6,8,365))*3,
                        np.exp(np.linspace(8,5,n-730))])
noise = np.random.normal(0, trend*0.15)
target = np.maximum(trend+noise, 0)

df = pd.DataFrame({"date": date_range})
df["new_cases"]        = target.astype(int)
df["new_cases_smooth"] = smooth_series(pd.Series(target))
df = mark_omicron(df)

FEATURE_COLS = ["temperature","humidity","precipitation","wind_speed",
                "transit_mobility","retail_mobility","workplaces_mobility",
                "stringency","new_tests","positive_rate",
                "query_symptoms","query_anosmia","query_cough","query_hospital","query_pcr"]

df["temperature"]         = np.sin(np.linspace(0,4*np.pi,n))*15 + np.random.normal(0,3,n)
df["humidity"]            = np.random.uniform(50,90,n)
df["precipitation"]       = np.abs(np.random.normal(2,4,n))
df["wind_speed"]          = np.abs(np.random.normal(5,3,n))
df["transit_mobility"]    = -trend/trend.max()*30 + np.random.normal(0,5,n)
df["retail_mobility"]     = -trend/trend.max()*20 + np.random.normal(0,5,n)
df["workplaces_mobility"] = -trend/trend.max()*25 + np.random.normal(0,5,n)
df["stringency"]          = trend/trend.max()*60 + np.random.normal(0,5,n)
df["new_tests"]           = trend/trend.max()*5000 + np.random.normal(0,200,n)
df["positive_rate"]       = trend/trend.max()*0.3 + np.random.normal(0,0.02,n)
df["query_symptoms"]      = target * np.random.uniform(0.8,1.2,n)
df["query_anosmia"]       = target * np.random.uniform(0.4,0.8,n)
df["query_cough"]         = target * np.random.uniform(0.3,0.6,n)
df["query_hospital"]      = target * np.random.uniform(0.2,0.5,n)
df["query_pcr"]           = target * np.random.uniform(0.5,0.9,n)

df = shift_features(df, FEATURE_COLS, shift=SHIFT)
df = fill_missing(df)
print(f"Признаков: {len(FEATURE_COLS)}")
print(f"Датасет: {df.shape}")

## 2. Feature Selection — RFE

In [ ]:
TARGET = "new_cases_smooth"
train_df, test_df = train_test_split_temporal(df.dropna(), TEST_DAYS)
X_train = train_df[FEATURE_COLS].values
y_train = train_df[TARGET].values
X_test  = test_df[FEATURE_COLS].values
y_test  = test_df[TARGET].values

N_FEATURES = 8
rfe = RFE(estimator=LinearRegression(), n_features_to_select=N_FEATURES)
rfe.fit(X_train, y_train)
selected = [col for col, sup in zip(FEATURE_COLS, rfe.support_) if sup]
print(f"Выбранные признаки RFE (n={N_FEATURES}):")
for f in selected: print(f"  ✓ {f}")
X_train_sel = X_train[:, rfe.support_]
X_test_sel  = X_test[:, rfe.support_]

## 3. Train Data Reduction

In [ ]:
sizes = list(range(100, len(train_df), 20))
rmse_scores = []
xgb_ref = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                        subsample=0.8, random_state=42, verbosity=0)
for n in sizes:
    X_tr = X_train_sel[-n:]
    y_tr = y_train[-n:]
    xgb_ref.fit(X_tr, y_tr)
    pred = xgb_ref.predict(X_test_sel)
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, pred)))

best_n = sizes[np.argmin(rmse_scores)]
print(f"Оптимальный размер обучающей выборки: {best_n} дней")
plot_train_reduction(sizes, rmse_scores, save_path=f"{FIGURES}/03_train_reduction.png")
plt.show()

## 4. Финальные модели

In [ ]:
X_tr_final = X_train_sel[-best_n:]
y_tr_final = y_train[-best_n:]

lr = LinearRegression()
lr.fit(X_tr_final, y_tr_final)
lr_pred = lr.predict(X_test_sel)
m_lr = compute_metrics(y_test, lr_pred, "Linear Regression")

rf = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_tr_final, y_tr_final)
rf_pred = rf.predict(X_test_sel)
m_rf = compute_metrics(y_test, rf_pred, "Random Forest")

xgb = XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=5,
                   subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
xgb.fit(X_tr_final, y_tr_final)
xgb_pred = xgb.predict(X_test_sel)
m_xgb = compute_metrics(y_test, xgb_pred, "XGBoost")

all_metrics = metrics_table([m_lr, m_rf, m_xgb])
all_metrics.to_csv(f"{METRICS}/03_multivariate_metrics.csv", index=False)
print(all_metrics.to_string(index=False))

## 5. Визуализация

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(test_df["date"].values, y_test, label="Факт", color="#2c3e50", linewidth=2)
ax.plot(test_df["date"].values, lr_pred,  label="LR",     linestyle="--", alpha=0.7)
ax.plot(test_df["date"].values, rf_pred,  label="RF",     linestyle="--", alpha=0.7)
ax.plot(test_df["date"].values, xgb_pred, label="XGBoost",color="#e74c3c", linewidth=1.5)
ax.set_ylabel("Новые случаи (сглаженные)")
ax.set_title("Multivariate: сравнение прогнозов")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES}/03_forecast_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
plot_metrics_bar(all_metrics, "RMSE", save_path=f"{FIGURES}/03_multivariate_rmse.png")
plt.show()

## 6. Вывод
> Мультивариантная модель значительно превосходит univariate baseline.
> Косвенные факторы несут реальную прогностическую информацию.

**Следующий шаг** → `04_xai.ipynb`